# Notebook 01: Download and Merge NHANES Data

## Purpose

Download the required public NHANES August 2021--August 2023 files, validate and merge them by participant identifier, construct project variables and the two operational targets, and create the adult analysis base and complete-case sample.

## Inputs

- Official NHANES XPT files: DEMO_L, DIQ_L, GHB_L, BMX_L, and HIQ_L

## Outputs

- `data/processed/nhanes_diabetes_analysis_base.csv`
- `data/processed/nhanes_diabetes_complete_case.csv`
- `data/processed/sample_metadata.json`
- Data dictionary and missingness summaries

## Dependencies

No upstream notebook dependency. Internet access is required only when the raw NHANES files are not already present.

> **Repository policy:** Notebook outputs and execution counts are cleared in the public source files. Run the notebooks in the documented order to regenerate all results.

## 1. Setup

In [ ]:
from pathlib import Path
import json
import requests
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

RANDOM_STATE = 26


## 2. Project folders

The project keeps raw data, processed participant-level data, and aggregate outputs separate.

In [ ]:
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
OUTPUT_DIR = PROJECT_DIR / "outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"

for folder in [RAW_DIR, PROCESSED_DIR, OUTPUT_DIR, FIGURE_DIR, TABLE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project directory: ", PROJECT_DIR)
print("Raw data:         ", RAW_DIR)
print("Processed data:   ", PROCESSED_DIR)
print("Tables:           ", TABLE_DIR)
print("Figures:          ", FIGURE_DIR)


## 3. Define and download the NHANES files

The notebook downloads files only when they are not already present locally.

In [ ]:
BASE_URL = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles"

NHANES_FILES = {
    "demo": "DEMO_L.xpt",  # demographics, income, weights, survey design
    "diq": "DIQ_L.xpt",    # diabetes questionnaire and treatment
    "ghb": "GHB_L.xpt",    # glycohemoglobin / HbA1c
    "bmx": "BMX_L.xpt",    # body measurements / BMI
    "hiq": "HIQ_L.xpt",    # health-insurance variables
}


def download_file(filename: str, raw_dir: Path = RAW_DIR) -> Path:
    """Download one NHANES XPT file unless it already exists locally."""
    url = f"{BASE_URL}/{filename}"
    local_path = raw_dir / filename

    if local_path.exists():
        print(f"Already present: {filename}")
        return local_path

    print(f"Downloading: {filename}")
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    local_path.write_bytes(response.content)
    print(f"Saved: {local_path}")
    return local_path


file_paths = {
    name: download_file(filename)
    for name, filename in NHANES_FILES.items()
}


## 4. Read and inspect the source files

In [ ]:
def read_xpt(path: Path) -> pd.DataFrame:
    """Read a SAS transport file and standardise column names to upper case."""
    table = pd.read_sas(path, format="xport")
    table.columns = table.columns.str.upper()
    return table


data = {
    name: read_xpt(path)
    for name, path in file_paths.items()
}

source_file_summary = pd.DataFrame(
    {
        "file": name,
        "rows": len(table),
        "columns": table.shape[1],
        "unique_seqn": table["SEQN"].nunique(),
        "seqn_is_unique": table["SEQN"].is_unique,
    }
    for name, table in data.items()
)

source_file_summary


## 5. Validate and merge the files

`validate="one_to_one"` prevents an accidental many-to-many merge from silently duplicating participants.

In [ ]:
for name, table in data.items():
    if "SEQN" not in table.columns:
        raise KeyError(f"SEQN is missing from {name}.")
    if not table["SEQN"].is_unique:
        raise ValueError(f"{name} contains duplicate SEQN values.")

n_merged_all = len(data["demo"])

df_merged = (
    data["demo"]
    .merge(data["diq"], on="SEQN", how="left", validate="one_to_one")
    .merge(data["ghb"], on="SEQN", how="left", validate="one_to_one")
    .merge(data["bmx"], on="SEQN", how="left", validate="one_to_one")
    .merge(data["hiq"], on="SEQN", how="left", validate="one_to_one")
)

if len(df_merged) != n_merged_all:
    raise RuntimeError("The merge changed the number of demographic-file participants.")

print("Merged shape:", df_merged.shape)
df_merged.head()


## 6. Select and rename project variables

The notebook keeps survey weights and design variables because later descriptive analyses may use NHANES-weighted point estimates.

In [ ]:
variable_map = {
    # Identifier
    "SEQN": "id",

    # Demographics and socioeconomic variables
    "RIDAGEYR": "age",
    "RIAGENDR": "sex_code",
    "RIDRETH3": "race_ethnicity_code",
    "INDFMPIR": "income_poverty_ratio",
    "RIDEXPRG": "pregnancy_status_code",

    # Survey participation, weight, and design variables
    "RIDSTATR": "exam_status",
    "WTMEC2YR": "mec_exam_weight",
    "SDMVSTRA": "survey_stratum",
    "SDMVPSU": "survey_psu",

    # Diabetes questionnaire
    "DIQ010": "doctor_diabetes_code",
    "DIQ180": "blood_test_past_3y_code",
    "DIQ050": "insulin_now_code",
    "DIQ070": "diabetes_pills_code",

    # Laboratory and body measurement
    "LBXGH": "hba1c",
    "BMXBMI": "bmi",

    # Health insurance
    "HIQ011": "health_insurance_code",
    "HIQ210": "uninsured_past_12mo_code",
}

missing_source_variables = [
    variable for variable in variable_map
    if variable not in df_merged.columns
]

if missing_source_variables:
    raise KeyError(
        "The following expected NHANES variables are missing: "
        f"{missing_source_variables}"
    )

selected_source_variables = list(variable_map)
df_project = (
    df_merged[selected_source_variables]
    .rename(columns=variable_map)
    .copy()
)

print("Selected-data shape:", df_project.shape)
df_project.head()


## 7. Restrict the project to adults

The adult restriction is applied before constructing the analytical variables.

In [ ]:
n_before_adult_restriction = len(df_project)

df_project = df_project.loc[df_project["age"] >= 18].copy()
n_adults = len(df_project)

# SAS/XPT missing numeric values can occasionally be imported as extremely
# small positive floating-point numbers, such as 5.397605e-79, instead of
# ordinary NaN values. These placeholders must be removed before targets,
# missingness, and complete cases are constructed. Exact zero values are
# preserved because zero is a possible income-to-poverty ratio.
sas_placeholder_columns = [
    "income_poverty_ratio",
    "bmi",
    "hba1c",
    "mec_exam_weight",
]

sas_placeholder_counts = {}

for column in sas_placeholder_columns:
    placeholder_mask = (
        df_project[column].notna()
        & df_project[column].gt(0)
        & df_project[column].lt(1e-50)
    )

    sas_placeholder_counts[column] = int(placeholder_mask.sum())
    df_project[column] = df_project[column].mask(placeholder_mask)

# A valid MEC examination weight is available only for participants who
# completed the MEC examination.
df_project["mec_exam_weight"] = df_project["mec_exam_weight"].mask(
    df_project["exam_status"] != 2
)

# Store identifiers and survey-design codes as nullable integers rather than
# floating-point values inherited from the XPT files.
integer_columns = [
    "id",
    "exam_status",
    "survey_stratum",
    "survey_psu",
]

for column in integer_columns:
    df_project[column] = df_project[column].astype("Int64")

print("All merged participants:", n_before_adult_restriction)
print("Adults aged 18+:", n_adults)
print("Removed SAS/XPT numeric missing-value placeholders:")
print(pd.Series(sas_placeholder_counts, name="n_removed"))
print(
    "Adults with a valid MEC examination weight:",
    int(df_project["mec_exam_weight"].notna().sum()),
)


## 8. Recode demographics and define primary-sample pregnancy eligibility

`RIDEXPRG` records pregnancy status at the MEC examination where it was
assessed. Missing values are often structural because the variable is not
applicable to every participant.

The primary analysis excludes only participants coded as **currently
pregnant**. Participants coded as not pregnant, indeterminate, or not assessed
remain eligible at this stage. This avoids treating structural missingness as
evidence of pregnancy.

In [ ]:
sex_map = {
    1.0: "Male",
    2.0: "Female",
}

race_map = {
    1.0: "Mexican American",
    2.0: "Other Hispanic",
    3.0: "Non-Hispanic White",
    4.0: "Non-Hispanic Black",
    6.0: "Non-Hispanic Asian",
    7.0: "Other or multiracial",
}

pregnancy_status_map = {
    1.0: "Pregnant",
    2.0: "Not pregnant",
    3.0: "Could not be determined",
}

df_project["sex"] = df_project["sex_code"].map(sex_map).astype("string")
df_project["race_ethnicity"] = (
    df_project["race_ethnicity_code"]
    .map(race_map)
    .astype("string")
)

pregnancy_status = (
    df_project["pregnancy_status_code"]
    .map(pregnancy_status_map)
    .astype("string")
    .fillna("Not assessed or structurally missing")
)

pregnancy_status_categories = [
    "Pregnant",
    "Not pregnant",
    "Could not be determined",
    "Not assessed or structurally missing",
]

df_project["pregnancy_status"] = pd.Categorical(
    pregnancy_status,
    categories=pregnancy_status_categories,
    ordered=False,
)

df_project["confirmed_current_pregnancy"] = (
    df_project["pregnancy_status_code"].eq(1.0).astype("Int64")
)

df_project["primary_sample_eligible"] = (
    1 - df_project["confirmed_current_pregnancy"]
).astype("Int64")

print("Sex:")
print(df_project["sex"].value_counts(dropna=False))
print("\nRace/ethnicity:")
print(df_project["race_ethnicity"].value_counts(dropna=False))
print("\nPregnancy status:")
print(df_project["pregnancy_status"].value_counts(dropna=False))
print(
    "\nConfirmed current pregnancies excluded from the primary sample:",
    int(df_project["confirmed_current_pregnancy"].sum()),
)


## 9. Construct the two operational diabetes targets

- `self_reported_prior_diagnosis` represents a participant's report that a doctor or health professional previously told them they had diabetes.
- `current_hba1c_ge_6_5` represents whether the participant's current measured HbA1c is at least 6.5%.

Neither target is labelled as the unique clinical gold standard.

In [ ]:
def recode_yes_no(series: pd.Series) -> pd.Series:
    """Map NHANES 1=Yes and 2=No to nullable binary values."""
    return series.map({1.0: 1, 2.0: 0}).astype("Int64")


df_project["self_reported_prior_diagnosis"] = recode_yes_no(
    df_project["doctor_diabetes_code"]
)

hba1c_target = pd.Series(
    pd.NA,
    index=df_project.index,
    dtype="Int64",
)
hba1c_observed = df_project["hba1c"].notna()
hba1c_target.loc[hba1c_observed] = (
    df_project.loc[hba1c_observed, "hba1c"] >= 6.5
).astype(int)

df_project["current_hba1c_ge_6_5"] = hba1c_target

print("Self-reported prior diagnosis:")
print(df_project["self_reported_prior_diagnosis"].value_counts(dropna=False))
print("\nCurrent HbA1c ≥ 6.5%:")
print(df_project["current_hba1c_ge_6_5"].value_counts(dropna=False))


## 10. Construct insurance variables

`HIQ210` is meaningful primarily for participants who are currently insured. A combined three-level variable prevents currently uninsured participants from being dropped simply because the past-year-gap question is skipped for them.

In [ ]:
df_project["currently_insured"] = recode_yes_no(
    df_project["health_insurance_code"]
)

insurance_history = pd.Series(
    pd.NA,
    index=df_project.index,
    dtype="string",
)

insurance_history.loc[
    df_project["health_insurance_code"] == 2
] = "Currently uninsured"

insurance_history.loc[
    (df_project["health_insurance_code"] == 1)
    & (df_project["uninsured_past_12mo_code"] == 1)
] = "Currently insured, past-year gap"

insurance_history.loc[
    (df_project["health_insurance_code"] == 1)
    & (df_project["uninsured_past_12mo_code"] == 2)
] = "Continuously insured"

insurance_categories = [
    "Continuously insured",
    "Currently insured, past-year gap",
    "Currently uninsured",
]

df_project["insurance_history"] = pd.Categorical(
    insurance_history,
    categories=insurance_categories,
    ordered=False,
)

print(df_project["currently_insured"].value_counts(dropna=False))
print()
print(df_project["insurance_history"].value_counts(dropna=False))


## 11. Construct screening and treatment variables

These variables are retained for descriptive interpretation of discordant label groups. They will **not** be used as model predictors because treatment follows diagnosis and could introduce leakage.

In [ ]:
df_project["blood_test_past_3y"] = recode_yes_no(
    df_project["blood_test_past_3y_code"]
)

df_project["insulin_now"] = recode_yes_no(
    df_project["insulin_now_code"]
)

df_project["diabetes_pills_now"] = recode_yes_no(
    df_project["diabetes_pills_code"]
)

any_medication = pd.Series(
    pd.NA,
    index=df_project.index,
    dtype="Int64",
)

diagnosed_mask = df_project["self_reported_prior_diagnosis"] == 1

any_medication.loc[
    diagnosed_mask
    & (
        (df_project["insulin_now"] == 1)
        | (df_project["diabetes_pills_now"] == 1)
    )
] = 1

any_medication.loc[
    diagnosed_mask
    & (df_project["insulin_now"] == 0)
    & (df_project["diabetes_pills_now"] == 0)
] = 0

df_project["any_diabetes_medication"] = any_medication

print("Blood test in past three years:")
print(df_project["blood_test_past_3y"].value_counts(dropna=False))
print("\nAny diabetes medication among diagnosed participants:")
print(
    df_project.loc[diagnosed_mask, "any_diabetes_medication"]
    .value_counts(dropna=False)
)


## 12. Basic data-quality checks

These checks detect impossible values and identifier problems before processed data are saved.

In [ ]:
if not df_project["id"].is_unique:
    raise ValueError("Participant IDs are not unique after merging.")

placeholder_check_columns = [
    "income_poverty_ratio",
    "bmi",
    "hba1c",
    "mec_exam_weight",
]

for column in placeholder_check_columns:
    remaining_placeholder_mask = (
        df_project[column].notna()
        & df_project[column].gt(0)
        & df_project[column].lt(1e-50)
    )

    if remaining_placeholder_mask.any():
        raise ValueError(
            f"{column} still contains an SAS/XPT numeric missing-value "
            "placeholder."
        )


valid_observed_pregnancy_codes = {1.0, 2.0, 3.0}
observed_pregnancy_codes = set(
    df_project["pregnancy_status_code"].dropna().unique()
)

if not observed_pregnancy_codes.issubset(valid_observed_pregnancy_codes):
    raise ValueError(
        "Unexpected RIDEXPRG pregnancy-status code detected: "
        f"{sorted(observed_pregnancy_codes)}"
    )

if not set(
    df_project["confirmed_current_pregnancy"].dropna().unique()
).issubset({0, 1}):
    raise ValueError(
        "confirmed_current_pregnancy contains values other than 0 and 1."
    )

if not set(
    df_project["primary_sample_eligible"].dropna().unique()
).issubset({0, 1}):
    raise ValueError(
        "primary_sample_eligible contains values other than 0 and 1."
    )

if not (
    df_project["confirmed_current_pregnancy"]
    + df_project["primary_sample_eligible"]
).eq(1).all():
    raise ValueError(
        "Pregnancy exclusion and primary-sample eligibility are inconsistent."
    )

if not df_project["age"].dropna().between(18, 80).all():
    raise ValueError("Unexpected adult age value detected.")

if not df_project["bmi"].dropna().gt(0).all():
    raise ValueError("BMI contains a non-positive observed value.")

if not df_project["hba1c"].dropna().gt(0).all():
    raise ValueError("HbA1c contains a non-positive observed value.")

if not df_project["income_poverty_ratio"].dropna().between(0, 5).all():
    raise ValueError(
        "Income-to-poverty ratio lies outside the documented range 0–5."
    )

if not df_project["mec_exam_weight"].dropna().gt(0).all():
    raise ValueError("A non-positive MEC examination weight was detected.")

if df_project.loc[
    df_project["exam_status"] != 2,
    "mec_exam_weight",
].notna().any():
    raise ValueError(
        "A participant without a completed MEC examination has a valid "
        "MEC examination weight."
    )

quality_check = pd.Series(
    {
        "duplicate_ids": int(df_project["id"].duplicated().sum()),
        "missing_age": int(df_project["age"].isna().sum()),
        "missing_sex": int(df_project["sex"].isna().sum()),
        "missing_race_ethnicity": int(
            df_project["race_ethnicity"].isna().sum()
        ),
        "confirmed_current_pregnancy_n": int(
            df_project["confirmed_current_pregnancy"].sum()
        ),
        "primary_sample_eligible_adults": int(
            df_project["primary_sample_eligible"].sum()
        ),
        "removed_income_poverty_placeholders": int(
            sas_placeholder_counts["income_poverty_ratio"]
        ),
        "removed_bmi_placeholders": int(
            sas_placeholder_counts["bmi"]
        ),
        "removed_hba1c_placeholders": int(
            sas_placeholder_counts["hba1c"]
        ),
        "removed_mec_weight_placeholders": int(
            sas_placeholder_counts["mec_exam_weight"]
        ),
        "examined_adults": int((df_project["exam_status"] == 2).sum()),
        "adults_with_valid_mec_weight": int(
            df_project["mec_exam_weight"].notna().sum()
        ),
    },
    name="value",
)

quality_check


## 13. Create the adult analysis base

The analysis base retains **all adults**, including confirmed current
pregnancies and participants with missing targets or predictors. This allows
later notebooks to document the pregnancy exclusion, participant flow, and
included-versus-excluded differences transparently.

The primary complete-case sample is constructed later from adults with
`primary_sample_eligible == 1`.

In [ ]:
analysis_base_columns = [
    # Identifier
    "id",

    # Predictors and subgroup variables
    "age",
    "sex",
    "race_ethnicity",
    "pregnancy_status",
    "confirmed_current_pregnancy",
    "primary_sample_eligible",
    "income_poverty_ratio",
    "bmi",
    "currently_insured",
    "insurance_history",

    # Raw biomarker and two targets
    "hba1c",
    "self_reported_prior_diagnosis",
    "current_hba1c_ge_6_5",

    # Screening and treatment variables for descriptive analyses
    "blood_test_past_3y",
    "insulin_now",
    "diabetes_pills_now",
    "any_diabetes_medication",

    # Survey participation, weights, and design
    "exam_status",
    "mec_exam_weight",
    "survey_stratum",
    "survey_psu",
]

analysis_base = df_project[analysis_base_columns].copy()

print("Adult analysis-base shape:", analysis_base.shape)
analysis_base.head()


## 14. Inspect missingness

In [ ]:
missingness_table = (
    analysis_base
    .isna()
    .agg(["sum", "mean"])
    .T
    .rename(columns={"sum": "missing_count", "mean": "missing_share"})
    .sort_values("missing_share", ascending=False)
)

missingness_table


## 15. Construct the matched complete-case modelling sample

The same participants and the same predictors will be used for both targets.
HbA1c itself and treatment variables are deliberately excluded from the
predictor set.

The primary sample excludes only participants with confirmed current
pregnancy. Missing or indeterminate pregnancy status is not used as a
complete-case requirement.

In [ ]:
predictor_columns = [
    "age",
    "sex",
    "race_ethnicity",
    "income_poverty_ratio",
    "bmi",
    "insurance_history",
]

target_columns = [
    "self_reported_prior_diagnosis",
    "current_hba1c_ge_6_5",
]

required_survey_columns = [
    "exam_status",
    "mec_exam_weight",
    "survey_stratum",
    "survey_psu",
]

required_complete_case_columns = (
    predictor_columns
    + target_columns
    + required_survey_columns
)

complete_case = (
    analysis_base
    .loc[
        (analysis_base["primary_sample_eligible"] == 1)
        & (analysis_base["exam_status"] == 2)
    ]
    .dropna(subset=required_complete_case_columns)
    .copy()
    .reset_index(drop=True)
)

print("Adult analysis base:", len(analysis_base))
print(
    "Confirmed current pregnancies excluded:",
    int(analysis_base["confirmed_current_pregnancy"].sum()),
)
print(
    "Adults eligible before complete-case requirements:",
    int(analysis_base["primary_sample_eligible"].sum()),
)
print("Complete-case sample:", len(complete_case))
print("Share retained:", round(len(complete_case) / len(analysis_base), 4))


## 16. Final checkpoints before saving

In [ ]:
if complete_case["confirmed_current_pregnancy"].ne(0).any():
    raise ValueError(
        "The complete-case sample contains a confirmed current pregnancy."
    )

if not complete_case["primary_sample_eligible"].eq(1).all():
    raise ValueError(
        "The complete-case sample contains an ineligible participant."
    )

if not complete_case["exam_status"].eq(2).all():
    raise ValueError(
        "The complete-case sample contains a participant who was not "
        "examined in the MEC."
    )

if complete_case["mec_exam_weight"].isna().any():
    raise ValueError(
        "The complete-case sample contains missing MEC examination weights."
    )

if not complete_case["mec_exam_weight"].gt(0).all():
    raise ValueError(
        "The complete-case sample contains non-positive MEC examination "
        "weights."
    )

checkpoint = {
    "complete_case_shape": complete_case.shape,
    "duplicate_ids": int(complete_case["id"].duplicated().sum()),
    "confirmed_current_pregnancy_in_complete_case_n": int(
        complete_case["confirmed_current_pregnancy"].sum()
    ),
    "all_complete_cases_primary_sample_eligible": bool(
        complete_case["primary_sample_eligible"].eq(1).all()
    ),
    "remaining_missing_required_values": int(
        complete_case[required_complete_case_columns]
        .isna()
        .sum()
        .sum()
    ),
    "all_complete_cases_examined": bool(
        complete_case["exam_status"].eq(2).all()
    ),
    "all_complete_case_mec_weights_positive": bool(
        complete_case["mec_exam_weight"].gt(0).all()
    ),
    "minimum_complete_case_mec_weight": float(
        complete_case["mec_exam_weight"].min()
    ),
    "self_reported_prior_diagnosis_counts": (
        complete_case["self_reported_prior_diagnosis"]
        .value_counts()
        .sort_index()
        .to_dict()
    ),
    "current_hba1c_ge_6_5_counts": (
        complete_case["current_hba1c_ge_6_5"]
        .value_counts()
        .sort_index()
        .to_dict()
    ),
    "insurance_history_counts": (
        complete_case["insurance_history"]
        .value_counts(dropna=False)
        .to_dict()
    ),
}

checkpoint


## 17. Save processed datasets, metadata, and the data dictionary

Participant-level CSV files stay local and should remain excluded from GitHub. Aggregate tables and figures may be shared later.

In [ ]:
analysis_base_path = PROCESSED_DIR / "nhanes_diabetes_analysis_base.csv"
complete_case_path = PROCESSED_DIR / "nhanes_diabetes_complete_case.csv"
missingness_path = TABLE_DIR / "adult_analysis_base_missingness.csv"
metadata_path = PROCESSED_DIR / "sample_metadata.json"
data_dictionary_path = PROCESSED_DIR / "data_dictionary.csv"

analysis_base.to_csv(analysis_base_path, index=False)
complete_case.to_csv(complete_case_path, index=False)
missingness_table.to_csv(missingness_path)

n_confirmed_current_pregnancy = int(
    analysis_base["confirmed_current_pregnancy"].sum()
)
n_primary_eligible_adults = int(
    analysis_base["primary_sample_eligible"].sum()
)
eligible_mask = analysis_base["primary_sample_eligible"] == 1
n_examined_eligible_adults = int(
    (eligible_mask & analysis_base["exam_status"].eq(2)).sum()
)
n_both_targets = int(
    (
        eligible_mask
        & analysis_base["exam_status"].eq(2)
        & analysis_base[target_columns].notna().all(axis=1)
    ).sum()
)

sample_metadata = {
    "n_merged_all_ages": int(n_merged_all),
    "n_adults": int(len(analysis_base)),
    "n_confirmed_current_pregnancy_excluded": (
        n_confirmed_current_pregnancy
    ),
    "n_primary_eligible_adults": n_primary_eligible_adults,
    "n_examined_primary_eligible_adults": (
        n_examined_eligible_adults
    ),
    "n_adults_with_both_targets": n_both_targets,
    "n_complete_case": int(len(complete_case)),
    "predictor_columns": predictor_columns,
    "target_columns": target_columns,
    "required_survey_columns": required_survey_columns,
    "random_state": RANDOM_STATE,
}

with metadata_path.open("w", encoding="utf-8") as file:
    json.dump(sample_metadata, file, indent=2)

data_dictionary = pd.DataFrame(
    [
        ("id", "NHANES respondent sequence number", "identifier"),
        ("age", "Age in years at screening, top-coded at 80 years", "predictor/subgroup"),
        ("sex", "NHANES sex category", "predictor/subgroup"),
        ("race_ethnicity", "NHANES RIDRETH3 category", "predictor/subgroup"),
        ("pregnancy_status", "NHANES pregnancy status at the MEC examination; structural missingness retained as its own category", "sample eligibility/descriptive"),
        ("confirmed_current_pregnancy", "Indicator for confirmed current pregnancy", "primary-sample exclusion"),
        ("primary_sample_eligible", "Indicator equal to 1 unless current pregnancy is confirmed", "sample eligibility"),
        ("income_poverty_ratio", "Family income-to-poverty ratio, top-coded at 5", "predictor/subgroup"),
        ("bmi", "Body mass index", "predictor"),
        ("currently_insured", "Currently covered by health insurance", "descriptive subgroup"),
        ("insurance_history", "Current coverage combined with past-year coverage gap", "predictor/subgroup"),
        ("hba1c", "Current measured glycohemoglobin percentage", "descriptive biomarker"),
        ("self_reported_prior_diagnosis", "Self-reported prior clinician diagnosis of diabetes", "target"),
        ("current_hba1c_ge_6_5", "Indicator that current HbA1c is at least 6.5%", "target"),
        ("blood_test_past_3y", "Blood test for high blood sugar or diabetes in past three years", "descriptive"),
        ("insulin_now", "Currently taking insulin", "descriptive treatment"),
        ("diabetes_pills_now", "Currently taking diabetes pills", "descriptive treatment"),
        ("any_diabetes_medication", "Insulin or diabetes pills among diagnosed participants", "descriptive treatment"),
        ("exam_status", "NHANES interview/examination status", "survey variable"),
        ("mec_exam_weight", "Full-sample MEC examination weight", "survey weight"),
        ("survey_stratum", "Masked variance pseudo-stratum", "survey design"),
        ("survey_psu", "Masked variance pseudo-PSU", "survey design"),
    ],
    columns=["variable", "description", "role"],
)

data_dictionary.to_csv(data_dictionary_path, index=False)

print("Saved analysis base: ", analysis_base_path)
print("Saved complete case: ", complete_case_path)
print("Saved metadata:      ", metadata_path)
print("Saved data dictionary:", data_dictionary_path)
print("Saved missingness table:", missingness_path)


## Completion criteria

- The adult source sample and complete-case sample pass the documented row-count checks.
- Confirmed current pregnancy is handled as a prespecified eligibility exclusion.
- Processed datasets, metadata, and the data dictionary are written successfully.